###OPTIMIZE COMMAND IN DATABRICKS (DELTA LAKE)
####PURPOSE:
- The OPTIMIZE command in Databricks compacts small files into larger ones within a Delta table.
- This improves query performance by reducing the number of files that Spark needs to read.

####WHY NEEDED:
- Delta tables can accumulate many small files due to frequent updates, merges, and streaming writes.
- OPTIMIZE combines these small files into fewer large files, which helps improve read performance.
####HOW IT WORKS:
- It rewrites data files within each partition(if any) into optimized files.
- Uses a bin-packing algorithm to combine smaller files into target-sized files (~1GB each).
- Only affects physical layout of data — does NOT change data content.##

In [0]:
%sql
show catalogs

In [0]:
%sql
show schemas in new_catalog

In [0]:
%sql
CREATE OR REPLACE TABLE new_catalog.default_schema.tblsales
(
  sales_id INT,
  product_id INT,
  region STRING,
  sales_amount DOUBLE,
  sales_date DATE
)
USING DELTA;

In [0]:
%sql
-- Step 2: Insert sample data

-- Let’s add multiple small batches to simulate many small files:

INSERT INTO new_catalog.default_schema.tblsales VALUES
  (1, 101, 'North', 1000.50, '2025-10-16'),
  (2, 102, 'South', 500.75, '2025-10-16'),
  (3, 103, 'East', 700.20, '2025-10-16'),
  (4, 104, 'West', 1200.00, '2025-10-16');

INSERT INTO new_catalog.default_schema.tblsales VALUES
  (5, 101, 'North', 800.00, '2025-10-17'),
  (6, 102, 'South', 450.00, '2025-10-17'),
  (7, 103, 'East', 600.00, '2025-10-17'),
  (8, 104, 'West', 1100.00, '2025-10-17');

In [0]:
%sql
select * from new_catalog.default_schema.tblsales;
--spark.sql("select * from new_catalog.default_schema.tblsales");
--spark.Table("new_catalog.default_schema.tblsales")

In [0]:
%sql
--Check defragmentation
DESCRIBE DETAIL new_catalog.default_schema.tblsales

In [0]:
%sql
-- Optimize command improves query performance by compacting small files
-- Catalog: new_catalog
-- Schema: default_schema
-- Table: tblsales
-- This will reorganize data files for tblsales to reduce fragmentation
-- Recommended after heavy inserts/updates for better read efficiency

OPTIMIZE new_catalog.default_schema.tblsales;

In [0]:
%sql
-- DESCRIBE DETAIL provides metadata about a table
-- Catalog: new_catalog
-- Schema: default_schema
-- Table: tblsales
-- Useful for auditing, validation, and performance monitoring
DESCRIBE DETAIL new_catalog.default_schema.tblsales;